In [ ]:
# Submodule smoke test: import the package + walk every submodule.
import importlib
import pkgutil

import azure.mgmt.resource

print(f"azure.mgmt.resource.__path__ = {list(azure.mgmt.resource.__path__)}")

failures = []
for module_info in pkgutil.walk_packages(azure.mgmt.resource.__path__, prefix='azure.mgmt.resource.'):
    try:
        importlib.import_module(module_info.name)
    except Exception as e:
        failures.append((module_info.name, type(e).__name__, str(e)[:200]))

if failures:
    print(f"\n{len(failures)} submodule import(s) failed:")
    for name, exc_type, msg in failures[:200]:  # print up to 200 failures
        print(f"  {name}: {exc_type}: {msg}")
    raise AssertionError(f"{len(failures)} submodule(s) of azure.mgmt.resource failed to import")
else:
    print(f"\nAll submodules import cleanly")

In [ ]:
# Example 1: construct ResourceManagementClient offline (no network) + inspect op groups.
# azure-mgmt-resource 26.x split the top-level `azure.mgmt.resource` namespace into
# sub-clients; ResourceManagementClient now lives under `.resources`.
import datetime
from azure.mgmt.resource.resources import ResourceManagementClient
from azure.core.credentials import AccessToken

class MockCredential:
    def get_token(self, *scopes, **kwargs):
        return AccessToken("mock-token", int(datetime.datetime.now().timestamp()) + 3600)

client = ResourceManagementClient(
    credential=MockCredential(),
    subscription_id="00000000-0000-0000-0000-000000000000",
)
print(f"client: {type(client).__module__}.{type(client).__name__}")
print(f"api_version: {client._config.api_version}")

for op_group in ("resource_groups", "resources", "providers", "tags"):
    assert hasattr(client, op_group), f"missing operation group: {op_group}"
    print(f"  op group: {op_group} -> {type(getattr(client, op_group)).__name__}")
client.close()

In [ ]:
# Example 2: secondary model surface — GenericResource + Provider.
from azure.mgmt.resource.resources.models import GenericResource, Provider

resource = GenericResource(location="westus", kind="StorageV2", tags={"tier": "web"})
print(f"GenericResource: {resource.as_dict()}")
assert resource.kind == "StorageV2"
assert resource.location == "westus"
assert resource.tags["tier"] == "web"

provider = Provider(namespace="Microsoft.Compute")
print(f"Provider: {provider.as_dict()}")
assert provider.namespace == "Microsoft.Compute"

In [ ]:
# Example 3: nested model round-trip — Tags / TagsResource.
from azure.mgmt.resource.resources.models import Tags, TagsResource

tags = Tags(tags={"cost-center": "engineering", "environment": "prod"})
tags_resource = TagsResource(properties=tags)
serialized = tags_resource.as_dict()
print(f"TagsResource: {serialized}")

# Reconstruct from the serialized dict and assert the nested payload survives.
rebuilt = TagsResource(properties=Tags(tags=serialized["properties"]["tags"]))
assert rebuilt.as_dict() == serialized
assert rebuilt.properties.tags["cost-center"] == "engineering"
print("nested Tags round-trip OK")

In [ ]:
# Example 4: ResourceGroup serialization round-trip (data-integrity invariant).
# azure-core typespec models round-trip through as_dict() (wire form, camelCase) and
# reconstruct straight from that mapping — the exact path the transport layer uses to
# (de)serialize request/response bodies.
from azure.mgmt.resource.resources.models import ResourceGroup

original = ResourceGroup(
    location="centralus",
    managed_by="/subscriptions/sub-123",
    tags={"a": "1", "b": "2"},
)
payload = original.as_dict()            # wire form (camelCase keys, e.g. managedBy)
restored = ResourceGroup(payload)       # deserialize straight back from the mapping
print(f"round-trip payload: {payload}")

assert restored.as_dict() == payload
assert restored.managed_by == "/subscriptions/sub-123"   # snake_case attribute access
assert restored.tags == {"a": "1", "b": "2"}
print("Example 4: azure_mgmt_resource smoke test complete")